<a href="https://colab.research.google.com/github/noodlejacknetwork/Disaster-Prediction-2025/blob/main/Disaster_Analysis_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DISASTER EVENTS SEVERITY PREDICTION

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. LOAD DATA
try:
    df = pd.read_csv('unclean_dataset.csv')
    print("✅ Data loaded successfully!")
except FileNotFoundError:
    print("❌ Error: Please make sure 'unclean_dataset.csv' is uploaded to the same folder.")
    raise

# 2. PREPROCESSING
# Drop IDs and handle dates
if 'event_id' in df.columns:
    df = df.drop(columns=['event_id'])

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df = df.drop(columns=['date'])

# Log transform and One-Hot Encoding
df['affected_population'] = np.log1p(df['affected_population'])
df = pd.get_dummies(df, columns=['disaster_type', 'location'], drop_first=True)

# PCA for Total Impact Score
pca_features = ['severity_level', 'affected_population', 'infrastructure_damage_index']
x_pca = StandardScaler().fit_transform(df[pca_features])
pca = PCA(n_components=2)
pcs = pca.fit_transform(x_pca)
df['total_impact_score'] = pcs[:, 0]
df['response_pattern_score'] = pcs[:, 1]

# 3. MODEL TRAINING (Random Forest)
features = ['total_impact_score', 'response_pattern_score', 'estimated_economic_loss_usd', 'aid_provided', 'year', 'month', 'day_of_week']
ohe_cols = [c for c in df.columns if 'disaster_type_' in c or 'location_' in c]
features.extend(ohe_cols)

X = df[features]
y = df['is_major_disaster']

# Split data and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
model = RandomForestClassifier(n_estimators=260, max_depth=20, random_state=42)
model.fit(X_train, y_train)

# 4. RESULTS
y_pred = model.predict(X_test)
print("\n================ MODEL RESULTS ================")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))